<h2 style="color: #e3b6ffff;">
<strong> Modelo ARIMA para predecir tráfico</strong>
</h2>


Samantha Sánchez Tinoco

---------------------------------------------------------------------

<h3 style="color: #eed2ffff;">
<strong>Parte 1: Desarrollo Teórico del Modelo ARIMA (2,1,2)
</h3>


### Forma Canónica de un Modelo ARIMA (p,d,q)

$$
\phi(B)(1 - B)^d y_t = \theta(B)\varepsilon_t
$$

### Para un modelo ARIMA (2, 1, 2)

- $p = 2$
- $d = 1$
- $q = 2$

Entonces:

$$
(1 - \phi_1 B - \phi_2 B^2)(1 - B)y_t
=
(1 + \theta_1 B + \theta_2 B^2)\varepsilon_t
$$

Primero se aplica la diferencia de primer orden:

$$
(1 - B)y_t = y_t - y_{t-1}
$$

Después se define como:

$$
w_t = y_t - y_{t-1}
$$

Y lo sustituimos en el modelo

$$
(1 - \phi_1 B - \phi_2 B^2) w_t
=
(1 + \theta_1 B + \theta_2 B^2)\varepsilon_t
$$


Ahora expandemos:

$$
w_t - \phi_1 w_{t-1} - \phi_2 w_{t-2}
=
\varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2}
$$

Y despejamos $w_t$:

$$
w_t =
\phi_1 w_{t-1}
+ \phi_2 w_{t-2}
+ \varepsilon_t
+ \theta_1 \varepsilon_{t-1}
+ \theta_2 \varepsilon_{t-2}
$$

Para reemplazar de nuevo $w_t = y_t - y_{t-1}$:

$$
y_t - y_{t-1} =
\phi_1 (y_{t-1} - y_{t-2})
+ \phi_2 (y_{t-2} - y_{t-3})
+ \varepsilon_t
+ \theta_1 \varepsilon_{t-1}
+ \theta_2 \varepsilon_{t-2}
$$

### Interpretación del Modelo

El modelo ARIMA (2, 1, 2):

- Aplica una diferenciación de primer orden ($d=1$) para convertir una serie no estacionaria en estacionaria.
- También incluye dos términos autoregresivos ($p=2$), para tener dependencia de los dos periodos anteriores.
- Incluye dos términos de medias móviles ($q=2$), para capturar el efecto de eventos pasados.
- Y modela tanto continuidad temporal como los efectos acumulados del error.


### Conclusión

El modelo ARIMA(2,1,2):

- Nos permite modelar series inicialmente no estacionarias.
- Captura cambios y dinámicas de corto plazo en los datos.
- Integra memoria  efectos de ruido pasado.


--------------------------------------------

<h3 style="color: #eed2ffff;">
<strong>Parte 2: Modelado con 3 librerías diferentes
</h3>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA as ARIMA_sm
from pmdarima import ARIMA as ARIMA_pm
from sktime.forecasting.arima import ARIMA as ARIMA_sk
from sktime.forecasting.base import ForecastingHorizon
from sklearn.metrics import mean_absolute_error, mean_squared_error

ValueError: A distribution name is required.

In [ ]:
df = pd.read_csv("../Datasets/retail_forecasting_70000.csv", parse_dates=["Date"])
df = df.set_index("Date")

serie = df["Revenue"]

In [ ]:
# Serie temporal diaria

# Agrupar por fecha y sumar Revenue
serie_diaria = df.groupby('Date')['Revenue'].sum()

# Completar fechas faltantes
fecha_completa = pd.date_range(start=serie_diaria.index.min(), 
                              end=serie_diaria.index.max(), 
                              freq='D')
serie_diaria = serie_diaria.reindex(fecha_completa, fill_value=0)

print(f"Serie diaria creada:")
print(f"  - Registros: {len(serie_diaria)}")
print(f"  - Desde: {serie_diaria.index[0]}")
print(f"  - Hasta: {serie_diaria.index[-1]}")

Serie diaria creada:
  - Registros: 1826
  - Desde: 2018-01-01 00:00:00
  - Hasta: 2022-12-31 00:00:00


In [ ]:
# Dividir en entrenamiento y prueba usando el último mes como prueba
train = serie[serie.index < "2022-12-01"]
test = serie[serie.index >= "2022-12-01"]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(f"Train: {train.index[0]} a {train.index[-1]}")
print(f"Test: {test.index[0]} a {test.index[-1]}")


Train shape: (68803,)
Test shape: (1197,)
Train: 2022-03-20 00:00:00 a 2022-01-18 00:00:00
Test: 2022-12-22 00:00:00 a 2022-12-09 00:00:00


In [ ]:
# Prueba ADF en serie original
result = adfuller(train.dropna())
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")# Prueba ADF en serie original
result = adfuller(train.dropna())
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")


--- Serie Original ---
ADF Statistic: -84.0499
p-value: 0.0
La serie es estacionaria

Veces que se diferenció: 0


In [ ]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', name='Train', line=dict(color='blue')))
fig1.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', name='Test', line=dict(color='red')))
fig1.add_vline(x=pd.to_datetime("2023-12-01"), line_dash="dash", line_color="black")
fig1.update_layout(title="Volumen de Tráfico - Train/Test Split", xaxis_title="Fecha", yaxis_title="Volumen", width=900, height=450)
fig1.show()

#### Modelo con statsmodel

In [ ]:
# Entrenar
modelo1 = ARIMA_sm(train, order=(3, 1, 1))
modelo1_fit = modelo1.fit()
print(modelo1_fit.summary().tables[1])

# Predecir
pred1 = modelo1_fit.forecast(steps=len(test))

# Métricas
mae1 = mean_absolute_error(test, pred1)
rmse1 = np.sqrt(mean_squared_error(test, pred1))
print(f"\n STATSMODELS - MAE: {mae1:.2f}, RMSE: {rmse1:.2f}")

c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency h will be used.

c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency h will be used.

c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency h will be used.



                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.0731      0.012      6.149      0.000       0.050       0.096
ar.L2          0.0619      0.011      5.758      0.000       0.041       0.083
ar.L3          0.0529      0.011      4.781      0.000       0.031       0.075
ma.L1         -1.0000      0.005   -194.870      0.000      -1.010      -0.990
sigma2      5.237e+05   3113.509    168.204      0.000    5.18e+05     5.3e+05

 STATSMODELS - MAE: 471.78, RMSE: 712.08


In [ ]:
fig_sm = go.Figure()
fig_sm.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', name='Train', line=dict(color='blue')))
fig_sm.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', name='Test Real', line=dict(color='red')))
fig_sm.add_trace(go.Scatter(x=test.index, y=pred1, mode='lines', name='Predicción', line=dict(color='green', dash='dash')))
fig_sm.add_vline(x=pd.to_datetime("2023-12-01"), line_dash="dash", line_color="black")
fig_sm.update_layout(title=f"Statsmodels ARIMA(2,{d},1) - MAE: {mae1:.1f}", xaxis_title="Fecha", yaxis_title="Volumen", width=900, height=400)
fig_sm.show()

#### Modelo 2: PMDARIMA (Auto ARIMA)

In [ ]:
modelo2 = ARIMA_pm(order=(2, d, 1))
modelo2_fit = modelo2.fit(train)

# Predecir
pred2 = modelo2_fit.predict(n_periods=len(test))

# Métricas
mae2 = mean_absolute_error(test, pred2)
rmse2 = np.sqrt(mean_squared_error(test, pred2))
print(f"\n PMDARIMA - MAE: {mae2:.2f}, RMSE: {rmse2:.2f}")
print(f"Modelo seleccionado: {modelo2_fit.order}")


c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency h will be used.

c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency h will be used.




 PMDARIMA - MAE: 472.40, RMSE: 712.21
Modelo seleccionado: (2, 0, 1)


In [ ]:
fig_pm = go.Figure()
fig_pm.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', name='Train', line=dict(color='blue')))
fig_pm.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', name='Test Real', line=dict(color='red')))
fig_pm.add_trace(go.Scatter(x=test.index, y=pred2, mode='lines', name='Predicción', line=dict(color='orange', dash='dot')))
fig_pm.add_vline(x=pd.to_datetime("2023-12-01"), line_dash="dash", line_color="black")
fig_pm.update_layout(title=f"Pmdarima ARIMA{modelo2_fit.order} - MAE: {mae2:.1f}", xaxis_title="Fecha", yaxis_title="Volumen", width=900, height=400)
fig_pm.show()


#### Modelo 3: SkTime ARIMA

In [ ]:
# Preparar datos
y_train = pd.Series(train.values, index=pd.PeriodIndex(train.index, freq='h'))
y_test = pd.Series(test.values, index=pd.PeriodIndex(test.index, freq='h'))


In [ ]:
# Entrenar
modelo3 = ARIMA_sk(order=(2, d, 1))
modelo3_fit = modelo3.fit(y_train)

# Predecir
fh = ForecastingHorizon(y_test.index, is_relative=False)
pred3 = modelo3_fit.predict(fh)

# Métricas
mae3 = mean_absolute_error(y_test, pred3)
rmse3 = np.sqrt(mean_squared_error(y_test, pred3))
print(f"\n SKTIME - MAE: {mae3:.2f}, RMSE: {rmse3:.2f}")



 SKTIME - MAE: 472.40, RMSE: 712.21


In [ ]:
# Gráfico
fig_sk = go.Figure()
fig_sk.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', name='Train', line=dict(color='blue')))
fig_sk.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', name='Test Real', line=dict(color='red')))
fig_sk.add_trace(go.Scatter(x=test.index, y=pred3, mode='lines', name='Predicción', line=dict(color='purple', dash='dashdot')))
fig_sk.add_vline(x=pd.to_datetime("2023-12-01"), line_dash="dash", line_color="black")
fig_sk.update_layout(title=f"Sktime ARIMA(2,{d},1) - MAE: {mae3:.1f}", xaxis_title="Fecha", yaxis_title="Volumen", width=900, height=400)
fig_sk.show()

#### Comparación de resultados

In [ ]:
comparacion = pd.DataFrame({
    'Librería': ['Statsmodels', 'Pmdarima', 'Sktime'],
    'Modelo': [f'ARIMA(2,{d},1)', f'{modelo2_fit.order}', f'ARIMA(2,{d},1)'],
    'MAE': [mae1, mae2, mae3],
    'RMSE': [rmse1, rmse2, rmse3]
})
comparacion['MAE'] = comparacion['MAE'].round(2)
comparacion['RMSE'] = comparacion['RMSE'].round(2)

print("\nCOMPARACIÓN DE MODELOS:")
print(comparacion.to_string(index=False))


COMPARACIÓN DE MODELOS:
   Librería       Modelo    MAE   RMSE
Statsmodels ARIMA(2,0,1) 471.82 712.09
   Pmdarima    (2, 0, 1) 472.40 712.21
     Sktime ARIMA(2,0,1) 472.40 712.21
